# Reproduction notebook: 24_timemixer_official_baseline_reproduction_fixed_v2

This notebook is retained as an executable provenance record for the anonymous supplementary package. Saved outputs and internal development notes have been removed.


In [ ]:

from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 500)

BASE = Path(
    "/code/stock_regime_retrieval/"
    "strong_forecaster"
)

REPO = BASE / "TimeMixer_official"

ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "timemixer_official_baseline_reproduction"
)

CHECKPOINT_ROOT = ROOT / "checkpoints"
LOG_ROOT = ROOT / "logs"
ARTIFACT_ROOT = ROOT / "artifacts"

for p in [
    ROOT,
    CHECKPOINT_ROOT,
    LOG_ROOT,
    ARTIFACT_ROOT,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

SUMMARY_PATH = ROOT / "summary.csv"

DATASETS = [
    "ETTm1",
    "ETTh1",
    "Weather",
    "Electricity",
]

HORIZONS = [
    96,
    192,
    336,
    720,
]

TASKS = [
    (d, h)
    for d in DATASETS
    for h in HORIZONS
]

RESUME = True
FORCE = False

# The official run.py fixes seed=2021.
OFFICIAL_SEED = 2021

# Use the first visible GPU inside the notebook process.
GPU = 0

print("Repository target:", REPO)
print("Output:", ROOT)
print("Tasks:", len(TASKS))


print(
    "Note: older timemixerpp_* output folders are left untouched. "
    "This corrected notebook writes to timemixer_official_baseline_reproduction."
)


In [ ]:

if not (
    REPO
    / ".git"
).is_dir():
    REPO.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Cloning official repository..."
    )

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/kwuking/TimeMixer.git",
            str(
                REPO
            ),
        ],
        check=True,
    )
else:
    print(
        "Repository already exists:",
        REPO,
    )

commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(
            REPO
        ),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

branch = subprocess.check_output(
    [
        "git",
        "-C",
        str(
            REPO
        ),
        "rev-parse",
        "--abbrev-ref",
        "HEAD",
    ],
    text=True,
).strip()

print("Branch:", branch)
print("Commit:", commit)

(
    ARTIFACT_ROOT
    / "official_repo_commit.txt"
).write_text(
    commit
    + "\n"
)

(
    ARTIFACT_ROOT
    / "official_repo_branch.txt"
).write_text(
    branch
    + "\n"
)


In [ ]:

readme_path = (
    REPO
    / "README.md"
)

model_path = (
    REPO
    / "models"
    / "TimeMixer.py"
)

script_probe_path = (
    REPO
    / "scripts"
    / "long_term_forecast"
    / "ETT_script"
    / "TimeMixer_ETTm1_unify.sh"
)

for p in [
    readme_path,
    model_path,
    script_probe_path,
]:
    if not p.is_file():
        raise FileNotFoundError(
            p
        )

readme_text = readme_path.read_text(
    errors="ignore"
)

model_text = model_path.read_text(
    errors="ignore"
)

script_text = script_probe_path.read_text(
    errors="ignore"
)

signals = {
    "README_header_mentions_TimeMixer":
        "TimeMixer: Decomposable Multiscale Mixing"
        in readme_text,

    "Model_file_exists":
        model_path.is_file(),

    "Official_LTSF_script_uses_TimeMixer":
        "model_name=TimeMixer"
        in script_text,

    "README_also_mentions_TimeMixer++":
        "TimeMixer++"
        in readme_text,
}

display(
    pd.DataFrame([
        {
            "Signal":
                k,
            "Present":
                v,
        }
        for k, v
        in signals.items()
    ])
)

(
    ARTIFACT_ROOT
    / "model_identity_signals.json"
).write_text(
    json.dumps(
        signals,
        indent=2,
    )
    + "\n"
)

required_identity = [
    "README_header_mentions_TimeMixer",
    "Model_file_exists",
    "Official_LTSF_script_uses_TimeMixer",
]

if not all(
    signals[
        k
    ]
    for k in required_identity
):
    raise RuntimeError(
        "Official TimeMixer identity/provenance audit failed."
    )

print(
    "PASS: this experiment is labeled and executed as TimeMixer."
)


In [ ]:

DATA_SOURCE_CANDIDATES = {
    "ETTm1": [
        Path(
            "/data/dataset/ETTm1.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/ETT-small/ETTm1.csv"
        ),
        Path(
            "/data/Time-Series-Library/"
            "dataset/ETT-small/ETTm1.csv"
        ),
    ],

    "ETTh1": [
        Path(
            "/data/dataset/ETTh1.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/ETT-small/ETTh1.csv"
        ),
        Path(
            "/data/Time-Series-Library/"
            "dataset/ETT-small/ETTh1.csv"
        ),
    ],

    "Weather": [
        Path(
            "/data/dataset/weather.csv"
        ),
        Path(
            "/data/dataset/weather/weather.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/weather/weather.csv"
        ),
        Path(
            "/data/Time-Series-Library/"
            "dataset/weather/weather.csv"
        ),
    ],

    "Electricity": [
        Path(
            "/data/dataset/electricity.csv"
        ),
        Path(
            "/data/dataset/electricity/electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/electricity/electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library/"
            "dataset/electricity/electricity.csv"
        ),
    ],
}

REPO_DATA_TARGETS = {
    "ETTm1":
        REPO
        / "dataset"
        / "ETT-small"
        / "ETTm1.csv",

    "ETTh1":
        REPO
        / "dataset"
        / "ETT-small"
        / "ETTh1.csv",

    "Weather":
        REPO
        / "dataset"
        / "weather"
        / "weather.csv",

    "Electricity":
        REPO
        / "dataset"
        / "electricity"
        / "electricity.csv",
}

DATA_SOURCES = {}

for name, candidates in (
    DATA_SOURCE_CANDIDATES.items()
):
    src = next(
        (
            p
            for p in candidates
            if p.is_file()
        ),
        None,
    )

    if src is None:
        raise FileNotFoundError(
            f"{name}: dataset CSV not found."
        )

    DATA_SOURCES[
        name
    ] = src

    target = REPO_DATA_TARGETS[
        name
    ]

    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if target.exists() or target.is_symlink():
        print(
            f"{name:11s} | existing repo dataset: {target}"
        )
    else:
        os.symlink(
            src,
            target,
        )

        print(
            f"{name:11s} | "
            f"{src} -> {target}"
        )

display(
    pd.DataFrame([
        {
            "Dataset":
                name,
            "Source":
                str(
                    DATA_SOURCES[
                        name
                    ]
                ),
            "RepoTarget":
                str(
                    REPO_DATA_TARGETS[
                        name
                    ]
                ),
            "Rows":
                len(
                    pd.read_csv(
                        DATA_SOURCES[
                            name
                        ]
                    )
                ),
        }
        for name in DATASETS
    ])
)


In [ ]:

RECIPES = {
    "ETTm1": {
        "data":
            "ETTm1",
        "root_path":
            "./dataset/ETT-small/",
        "data_path":
            "ETTm1.csv",
        "model_id_prefix":
            "ETTm1_96",
        "enc_in":
            7,
        "dec_in":
            7,
        "c_out":
            7,
        "e_layers":
            2,
        "factor":
            1,
        "d_model":
            16,
        "d_ff":
            32,
        "batch_size":
            16,
        "learning_rate":
            0.01,
        "train_epochs":
            100,
        "patience":
            15,
    },

    "ETTh1": {
        "data":
            "ETTh1",
        "root_path":
            "./dataset/ETT-small/",
        "data_path":
            "ETTh1.csv",
        "model_id_prefix":
            "ETTh1_96",
        "enc_in":
            7,
        "dec_in":
            7,
        "c_out":
            7,
        "e_layers":
            2,
        "factor":
            1,
        "d_model":
            16,
        "d_ff":
            32,
        "batch_size":
            128,
        "learning_rate":
            0.01,
        "train_epochs":
            10,
        "patience":
            10,
    },

    "Weather": {
        "data":
            "custom",
        "root_path":
            "./dataset/weather/",
        "data_path":
            "weather.csv",
        "model_id_prefix":
            "weather_96",
        "enc_in":
            21,
        "dec_in":
            21,
        "c_out":
            21,
        "e_layers":
            3,
        "factor":
            3,
        "d_model":
            16,
        "d_ff":
            32,
        "batch_size":
            128,
        "learning_rate":
            0.01,
        "train_epochs":
            20,
        "patience":
            10,
    },

    "Electricity": {
        "data":
            "custom",
        "root_path":
            "./dataset/electricity/",
        "data_path":
            "electricity.csv",
        "model_id_prefix":
            "ECL_96",
        "enc_in":
            321,
        "dec_in":
            321,
        "c_out":
            321,
        "e_layers":
            3,
        "factor":
            3,
        "d_model":
            16,
        "d_ff":
            32,
        "batch_size":
            32,
        "learning_rate":
            0.01,
        "train_epochs":
            20,
        "patience":
            10,
    },
}

display(
    pd.DataFrame(
        RECIPES
    ).T
)


In [ ]:

SCRIPT_PATHS = {
    "ETTm1":
        REPO
        / "scripts"
        / "long_term_forecast"
        / "ETT_script"
        / "TimeMixer_ETTm1_unify.sh",

    "ETTh1":
        REPO
        / "scripts"
        / "long_term_forecast"
        / "ETT_script"
        / "TimeMixer_ETTh1_unify.sh",

    "Weather":
        REPO
        / "scripts"
        / "long_term_forecast"
        / "Weather_script"
        / "TimeMixer_unify.sh",

    "Electricity":
        REPO
        / "scripts"
        / "long_term_forecast"
        / "ECL_script"
        / "TimeMixer_unify.sh",
}

audit_rows = []

for name, p in SCRIPT_PATHS.items():
    if not p.is_file():
        raise FileNotFoundError(
            p
        )

    txt = p.read_text(
        errors="ignore"
    )

    r = RECIPES[
        name
    ]

    checks = {
        "model_name=TimeMixer":
            "model_name=TimeMixer"
            in txt,

        "seq_len=96":
            "seq_len=96"
            in txt,

        "learning_rate=0.01":
            "learning_rate=0.01"
            in txt,

        f"e_layers={r['e_layers']}":
            (
                f"e_layers={r['e_layers']}"
                in txt
            ),

        f"d_model={r['d_model']}":
            (
                f"d_model={r['d_model']}"
                in txt
            ),

        f"d_ff={r['d_ff']}":
            (
                f"d_ff={r['d_ff']}"
                in txt
            ),
    }

    audit_rows.append({
        "Dataset":
            name,
        "Script":
            str(
                p.relative_to(
                    REPO
                )
            ),
        "AllBasicChecksPass":
            all(
                checks.values()
            ),
        "Checks":
            json.dumps(
                checks
            ),
    })

audit_df = pd.DataFrame(
    audit_rows
)

display(
    audit_df
)

if not audit_df[
    "AllBasicChecksPass"
].all():
    raise RuntimeError(
        "Official script provenance audit failed."
    )

print(
    "PASS: encoded recipes match basic official-script settings."
)


In [ ]:

# ---------------------------------------------------------------------
# Safe dependency repair for the current research environment.
# ---------------------------------------------------------------------
#
# Important:
#   - Never install the repository's complete requirements.txt here.
#   - It pins very old torch/numpy/pandas versions.
#   - We only repair import-time packages needed by the current checkout.
#
# The most common upstream failure is sktime:
# data_provider/data_loader.py imports it even though ETT/custom LTSF
# experiments do not use the classification .ts reader.
# ---------------------------------------------------------------------

import re
import subprocess
import sys
import os

env = os.environ.copy()

env["PYTHONPATH"] = (
    str(REPO)
    + os.pathsep
    + env.get(
        "PYTHONPATH",
        "",
    )
)

PROBE_CODE = (
    "import torch; "
    "from exp.exp_long_term_forecasting "
    "import Exp_Long_Term_Forecast; "
    "from models import TimeMixer; "
    "print('IMPORT_OK')"
)


def run_import_probe():
    return subprocess.run(
        [
            sys.executable,
            "-c",
            PROBE_CODE,
        ],
        cwd=REPO,
        env=env,
        text=True,
        capture_output=True,
    )


def pip_install(*packages):
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        *packages,
    ]

    print(
        "\nRunning safe compatibility install:"
    )
    print(
        " ".join(
            cmd
        )
    )

    subprocess.run(
        cmd,
        check=True,
    )


# Map import module names to PyPI package names.
SAFE_MODULE_TO_PACKAGE = {
    "sktime":
        "sktime",

    "einops":
        "einops",

    "pywt":
        "PyWavelets",

    "sympy":
        "sympy",

    "tqdm":
        "tqdm",
}


installed_this_cell = []

for attempt in range(
    1,
    6,
):
    probe = run_import_probe()

    print(
        f"\nImport probe attempt {attempt}"
    )

    if probe.stdout.strip():
        print(
            probe.stdout
        )

    if probe.returncode == 0:
        print(
            "PASS: official TimeMixer long-term forecasting import works."
        )
        break

    print(
        "Upstream import stderr:"
    )
    print(
        probe.stderr
    )

    err = probe.stderr

    repair_package = None
    repair_reason = None

    # -------------------------------------------------------------
    # 1. sktime-specific incompatibility.
    #
    # Covers both:
    #   ModuleNotFoundError: No module named 'sktime'
    # and
    #   ImportError: cannot import name
    #   'load_from_tsfile_to_dataframe' from 'sktime.datasets'
    # -------------------------------------------------------------
    if "sktime" in err:
        repair_package = "sktime"
        repair_reason = (
            "The upstream data loader imports sktime unconditionally. "
            "Use the current Python-compatible sktime rather than the "
            "repository's obsolete pin."
        )

    # -------------------------------------------------------------
    # 2. Other simple missing modules used by the current model.
    # -------------------------------------------------------------
    if repair_package is None:
        m = re.search(
            r"No module named ['\"]([^'\"]+)['\"]",
            err,
        )

        if m:
            module = (
                m.group(
                    1
                )
                .split(
                    "."
                )[
                    0
                ]
            )

            if module in SAFE_MODULE_TO_PACKAGE:
                repair_package = SAFE_MODULE_TO_PACKAGE[
                    module
                ]

                repair_reason = (
                    f"Missing optional/import-time module: {module}"
                )

    if repair_package is None:
        raise RuntimeError(
            "Official TimeMixer import failed for a reason that is "
            "not on the safe dependency allow-list.\n\n"
            "The full upstream traceback is printed above. "
            "Do not run pip install -r requirements.txt. "
            "Inspect that traceback before changing the environment."
        )

    if repair_package in installed_this_cell:
        raise RuntimeError(
            f"Import still fails after installing/upgrading "
            f"{repair_package}. Full traceback is printed above."
        )

    print(
        "\nCompatibility repair:"
    )
    print(
        repair_reason
    )

    pip_install(
        repair_package
    )

    installed_this_cell.append(
        repair_package
    )

else:
    raise RuntimeError(
        "Import probe exceeded the maximum number of safe repair attempts."
    )


print(
    "\nPackages installed/upgraded by this cell:",
    (
        installed_this_cell
        if installed_this_cell
        else "none"
    ),
)


In [ ]:

# Record versions actually used for reproducibility.
import importlib.metadata as md

packages = [
    "torch",
    "numpy",
    "pandas",
    "scikit-learn",
    "sktime",
    "einops",
    "PyWavelets",
]

version_rows = []

for pkg in packages:
    try:
        version = md.version(
            pkg
        )
    except md.PackageNotFoundError:
        version = "not installed"

    version_rows.append({
        "Package":
            pkg,
        "Version":
            version,
    })

env_versions = pd.DataFrame(
    version_rows
)

display(
    env_versions
)

env_versions.to_csv(
    ARTIFACT_ROOT
    / "environment_versions.csv",
    index=False,
)


In [ ]:

TOOLS_PATH = (
    REPO
    / "utils"
    / "tools.py"
)

if not TOOLS_PATH.is_file():
    raise FileNotFoundError(
        TOOLS_PATH
    )

tools_before = TOOLS_PATH.read_text(
    errors="ignore"
)

patch_record = {
    "File":
        str(
            TOOLS_PATH
        ),
    "Replacement":
        "np.Inf -> np.inf",
    "OccurrencesBefore":
        tools_before.count(
            "np.Inf"
        ),
}

if "np.Inf" in tools_before:
    tools_after = tools_before.replace(
        "np.Inf",
        "np.inf",
    )

    TOOLS_PATH.write_text(
        tools_after
    )

    print(
        "Applied NumPy 2 compatibility patch:"
    )
    print(
        "  np.Inf -> np.inf"
    )
else:
    tools_after = tools_before

    print(
        "No np.Inf occurrence found; "
        "the repository may already be compatible."
    )

patch_record[
    "OccurrencesAfter"
] = tools_after.count(
    "np.Inf"
)

patch_record[
    "np_inf_occurrences"
] = tools_after.count(
    "np.inf"
)

(
    ARTIFACT_ROOT
    / "numpy2_compatibility_patch.json"
).write_text(
    json.dumps(
        patch_record,
        indent=2,
    )
    + "\n"
)

# Verify import and EarlyStopping construction in a fresh process.
compat_probe = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import numpy as np; "
            "from utils.tools import EarlyStopping; "
            "e=EarlyStopping(); "
            "print('NUMPY', np.__version__); "
            "print('EARLYSTOP_MIN', e.val_loss_min); "
            "print('COMPAT_OK')"
        ),
    ],
    cwd=REPO,
    env=env,
    text=True,
    capture_output=True,
)

print(
    compat_probe.stdout
)

if compat_probe.returncode != 0:
    print(
        compat_probe.stderr
    )

    raise RuntimeError(
        "NumPy compatibility probe still failed."
    )

print(
    "PASS: upstream EarlyStopping is NumPy-compatible."
)


In [ ]:

def setting_name(
    dataset,
    horizon,
):
    r = RECIPES[
        dataset
    ]

    model_id = (
        f"{r['model_id_prefix']}_{horizon}"
    )

    # Mirrors official run.py setting string:
    # task, model_id, comment, model, data,
    # sl, pl, dm, nh, el, dl, df, factor,
    # embed, distil, des, itr.
    return (
        "long_term_forecast_"
        f"{model_id}_"
        "none_"
        "TimeMixer_"
        f"{r['data']}_"
        "sl96_"
        f"pl{horizon}_"
        f"dm{r['d_model']}_"
        "nh8_"
        f"el{r['e_layers']}_"
        "dl1_"
        f"df{r['d_ff']}_"
        f"fc{r['factor']}_"
        "ebtimeF_"
        "dtTrue_"
        "Exp_0"
    )


def build_command(
    dataset,
    horizon,
):
    r = RECIPES[
        dataset
    ]

    model_id = (
        f"{r['model_id_prefix']}_{horizon}"
    )

    cmd = [
        sys.executable,
        "-u",
        "run.py",

        "--task_name",
        "long_term_forecast",

        "--is_training",
        "1",

        "--root_path",
        r[
            "root_path"
        ],

        "--data_path",
        r[
            "data_path"
        ],

        "--model_id",
        model_id,

        "--model",
        "TimeMixer",

        "--data",
        r[
            "data"
        ],

        "--features",
        "M",

        "--seq_len",
        "96",

        "--label_len",
        "0",

        "--pred_len",
        str(
            horizon
        ),

        "--e_layers",
        str(
            r[
                "e_layers"
            ]
        ),

        "--d_layers",
        "1",

        "--factor",
        str(
            r[
                "factor"
            ]
        ),

        "--enc_in",
        str(
            r[
                "enc_in"
            ]
        ),

        "--dec_in",
        str(
            r[
                "dec_in"
            ]
        ),

        "--c_out",
        str(
            r[
                "c_out"
            ]
        ),

        "--des",
        "Exp",

        "--itr",
        "1",

        "--d_model",
        str(
            r[
                "d_model"
            ]
        ),

        "--d_ff",
        str(
            r[
                "d_ff"
            ]
        ),

        "--batch_size",
        str(
            r[
                "batch_size"
            ]
        ),

        "--learning_rate",
        str(
            r[
                "learning_rate"
            ]
        ),

        "--train_epochs",
        str(
            r[
                "train_epochs"
            ]
        ),

        "--patience",
        str(
            r[
                "patience"
            ]
        ),

        "--down_sampling_layers",
        "3",

        "--down_sampling_method",
        "avg",

        "--down_sampling_window",
        "2",

        "--checkpoints",
        str(
            CHECKPOINT_ROOT
        ),

        "--gpu",
        str(
            GPU
        ),
    ]

    return cmd


for d, h in TASKS[:4]:
    print(
        d,
        h,
    )
    print(
        " ".join(
            build_command(
                d,
                h,
            )
        )
    )
    print(
        "Setting:",
        setting_name(
            d,
            h,
        ),
    )
    print()


In [ ]:

MSE_MAE_PATTERNS = [
    re.compile(
        r"mse[:=]\s*"
        r"([0-9eE+\-.]+)"
        r"\s*,?\s*"
        r"mae[:=]\s*"
        r"([0-9eE+\-.]+)",
        re.IGNORECASE,
    ),

    re.compile(
        r"mae[:=]\s*"
        r"([0-9eE+\-.]+)"
        r"\s*,?\s*"
        r"mse[:=]\s*"
        r"([0-9eE+\-.]+)",
        re.IGNORECASE,
    ),
]


def parse_test_metrics(
    text,
):
    matches = []

    for p_i, pat in enumerate(
        MSE_MAE_PATTERNS
    ):
        for m in pat.finditer(
            text
        ):
            if p_i == 0:
                mse = float(
                    m.group(
                        1
                    )
                )

                mae = float(
                    m.group(
                        2
                    )
                )
            else:
                mae = float(
                    m.group(
                        1
                    )
                )

                mse = float(
                    m.group(
                        2
                    )
                )

            matches.append(
                (
                    m.start(),
                    mse,
                    mae,
                )
            )

    if not matches:
        raise RuntimeError(
            "Could not parse final MSE/MAE from official log."
        )

    matches.sort(
        key=lambda x:
            x[
                0
            ]
    )

    _, mse, mae = matches[
        -1
    ]

    return (
        mse,
        mae,
    )


def parse_best_epoch(
    text,
):
    current_epoch = None
    best_epoch = None

    for line in text.splitlines():
        m = re.search(
            r"Epoch:\s*(\d+)",
            line,
        )

        if m:
            current_epoch = int(
                m.group(
                    1
                )
            )

        if (
            "Validation loss decreased"
            in line
            and current_epoch is not None
        ):
            best_epoch = current_epoch

    return best_epoch


In [ ]:

if (
    RESUME
    and SUMMARY_PATH.is_file()
):
    summary_df = pd.read_csv(
        SUMMARY_PATH
    )
else:
    summary_df = pd.DataFrame()


def is_completed(
    dataset,
    horizon,
):
    if (
        FORCE
        or not len(
            summary_df
        )
    ):
        return False

    hit = summary_df[
        (
            summary_df[
                "Dataset"
            ]
            == dataset
        )
        & (
            summary_df[
                "Horizon"
            ]
            == horizon
        )
    ]

    return len(
        hit
    ) == 1


rows = (
    summary_df.to_dict(
        "records"
    )
    if len(
        summary_df
    )
    else []
)


for dataset, horizon in TASKS:
    if is_completed(
        dataset,
        horizon,
    ):
        print(
            f"SKIP completed: "
            f"{dataset} H={horizon}"
        )
        continue

    print(
        "\n"
        + "="
        * 130
    )

    print(
        f"OFFICIAL BASELINE | "
        f"{dataset} | H={horizon}"
    )

    print(
        "="
        * 130
    )

    cmd = build_command(
        dataset,
        horizon,
    )

    setting = setting_name(
        dataset,
        horizon,
    )

    log_path = (
        LOG_ROOT
        / (
            f"{dataset}_H{horizon}.log"
        )
    )

    start = time.time()

    run_env = os.environ.copy()

    # Respect an externally assigned CUDA_VISIBLE_DEVICES if present.
    # Inside the visible set, official --gpu 0 is used.
    run_env[
        "PYTHONPATH"
    ] = (
        str(
            REPO
        )
        + os.pathsep
        + run_env.get(
            "PYTHONPATH",
            "",
        )
    )

    with log_path.open(
        "w"
    ) as log_file:
        process = subprocess.Popen(
            cmd,
            cwd=REPO,
            env=run_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        selected_lines = []

        for line in process.stdout:
            log_file.write(
                line
            )

            if any(
                token
                in line
                for token in [
                    ">>>>>>>",
                    "Epoch:",
                    "Validation loss decreased",
                    "EarlyStopping",
                    "mse:",
                    "mae:",
                ]
            ):
                print(
                    line.rstrip()
                )

            selected_lines.append(
                line
            )

        return_code = process.wait()

    runtime_min = (
        time.time()
        - start
    ) / 60.0

    if return_code != 0:
        # Surface the real upstream failure directly in the notebook.
        failed_text = log_path.read_text(
            errors="ignore"
        )

        failed_lines = failed_text.splitlines()

        print(
            "\n"
            + "!" * 120
        )
        print(
            f"OFFICIAL RUN FAILED | "
            f"{dataset} H={horizon}"
        )
        print(
            f"Log: {log_path}"
        )
        print(
            "Last 120 log lines:"
        )
        print(
            "-" * 120
        )
        print(
            "\n".join(
                failed_lines[
                    -120:
                ]
            )
        )
        print(
            "!" * 120
        )

        # Give a targeted message for the most common NumPy 2 issue.
        if (
            "np.Inf" in failed_text
            or (
                "numpy" in failed_text.lower()
                and "Inf" in failed_text
                and "AttributeError" in failed_text
            )
        ):
            raise RuntimeError(
                "Run failed because upstream TimeMixer used "
                "the removed NumPy alias np.Inf. "
                "Rerun the NumPy compatibility patch cell, "
                "then rerun this experiment cell."
            )

        raise RuntimeError(
            f"Official TimeMixer run failed for "
            f"{dataset} H={horizon}. "
            "The actual upstream traceback is printed above."
        )

    text = log_path.read_text(
        errors="ignore"
    )

    mse, mae = parse_test_metrics(
        text
    )

    best_epoch = parse_best_epoch(
        text
    )

    checkpoint = (
        CHECKPOINT_ROOT
        / setting
        / "checkpoint.pth"
    )

    if not checkpoint.is_file():
        raise FileNotFoundError(
            "Expected official checkpoint not found: "
            + str(
                checkpoint
            )
        )

    row = {
        "Dataset":
            dataset,
        "Horizon":
            horizon,
        "MSE":
            mse,
        "MAE":
            mae,
        "BestEpoch":
            (
                best_epoch
                if best_epoch is not None
                else np.nan
            ),
        "Setting":
            setting,
        "Checkpoint":
            str(
                checkpoint
            ),
        "Log":
            str(
                log_path
            ),
        "OfficialCommit":
            commit,
        "Seed":
            OFFICIAL_SEED,
        "RuntimeMinutes":
            runtime_min,
        "TrainEpochLimit":
            RECIPES[
                dataset
            ][
                "train_epochs"
            ],
        "Patience":
            RECIPES[
                dataset
            ][
                "patience"
            ],
        "BatchSize":
            RECIPES[
                dataset
            ][
                "batch_size"
            ],
        "LearningRate":
            RECIPES[
                dataset
            ][
                "learning_rate"
            ],
        "ELayers":
            RECIPES[
                dataset
            ][
                "e_layers"
            ],
    }

    rows = [
        r
        for r in rows
        if not (
            r.get(
                "Dataset"
            )
            == dataset
            and int(
                r.get(
                    "Horizon",
                    -1,
                )
            )
            == horizon
        )
    ]

    rows.append(
        row
    )

    summary_df = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "Dataset",
                "Horizon",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    summary_df.to_csv(
        SUMMARY_PATH,
        index=False,
    )

    print(
        f"FINAL | "
        f"MSE={mse:.6f} | "
        f"MAE={mae:.6f} | "
        f"best_epoch={best_epoch} | "
        f"{runtime_min:.2f} min"
    )

display(
    summary_df
)


In [ ]:

expected = pd.MultiIndex.from_product(
    [
        DATASETS,
        HORIZONS,
    ],
    names=[
        "Dataset",
        "Horizon",
    ],
).to_frame(
    index=False
)

present = summary_df[
    [
        "Dataset",
        "Horizon",
    ]
].drop_duplicates()

missing = expected.merge(
    present,
    on=[
        "Dataset",
        "Horizon",
    ],
    how="left",
    indicator=True,
)

missing = missing[
    missing[
        "_merge"
    ]
    == "left_only"
]

if len(
    missing
):
    display(
        missing
    )

    raise RuntimeError(
        "Not all 16 baseline conditions are complete."
    )

dup = (
    summary_df
    .groupby(
        [
            "Dataset",
            "Horizon",
        ]
    )
    .size()
    .reset_index(
        name="Count"
    )
)

dup = dup[
    dup[
        "Count"
    ]
    != 1
]

if len(
    dup
):
    display(
        dup
    )

    raise RuntimeError(
        "Duplicate baseline conditions found."
    )

print(
    "PASS: all 16 baseline conditions complete."
)


In [ ]:

compact = summary_df[
    [
        "Dataset",
        "Horizon",
        "MSE",
        "MAE",
        "BestEpoch",
        "RuntimeMinutes",
    ]
].copy()

display(
    compact
)

compact.to_csv(
    ROOT
    / "compact_results.csv",
    index=False,
)


In [ ]:

dataset_summary = (
    summary_df
    .groupby(
        "Dataset",
        as_index=False,
    )
    .agg(
        Conditions=(
            "Horizon",
            "size",
        ),
        MeanMSE=(
            "MSE",
            "mean",
        ),
        MeanMAE=(
            "MAE",
            "mean",
        ),
        TotalRuntimeMinutes=(
            "RuntimeMinutes",
            "sum",
        ),
        MeanRuntimeMinutes=(
            "RuntimeMinutes",
            "mean",
        ),
    )
)

dataset_summary[
    "TotalRuntimeHours"
] = (
    dataset_summary[
        "TotalRuntimeMinutes"
    ]
    / 60.0
)

display(
    dataset_summary
)

dataset_summary.to_csv(
    ROOT
    / "dataset_summary.csv",
    index=False,
)


In [ ]:

print(
    "Output root:",
    ROOT,
)

for p in sorted(
    ROOT.rglob(
        "*"
    )
):
    if p.is_file():
        print(
            p.relative_to(
                ROOT
            )
        )
